# Atelier — EDA & Model Evaluation

Exploratory analysis and model comparison for the clothing review recommendation classifier.

**Dataset:** 19,662 women's e-commerce clothing reviews  
**Target:** `Recommended IND` (1 = would recommend, 0 = would not)

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from ml import compare_models, evaluate_holdout, prepare_training_frame
from nlp import preprocess

sns.set_theme(style='whitegrid')
df = pd.read_csv(ROOT / 'assignment3_II.csv')
print(f'Reviews: {len(df):,}')

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

df['Recommended IND'].value_counts().sort_index().plot.bar(ax=axes[0], color=['#e74c3c', '#2ecc71'])
axes[0].set_title('Class Balance')
axes[0].set_xlabel('Recommended')

df['Rating'].value_counts().sort_index().plot.bar(ax=axes[1], color='#3b82f6')
axes[1].set_title('Rating Distribution')
axes[1].set_xlabel('Stars')

text_lengths = (df['Title'].fillna('') + ' ' + df['Review Text'].fillna('')).str.len()
axes[2].hist(text_lengths, bins=40, color='#64748b')
axes[2].set_title('Review Text Length')
axes[2].set_xlabel('Characters')

plt.tight_layout()
plt.show()

print('Recommendation rate:', f"{df['Recommended IND'].mean():.1%}")
print('Median text length:', int(text_lengths.median()), 'characters')

In [ ]:
frame = prepare_training_frame(df)
frame['token_count'] = frame['processed'].str.split().str.len()

fig, ax = plt.subplots(figsize=(8, 4))
for label, color in [(0, '#e74c3c'), (1, '#2ecc71')]:
    subset = frame[frame['Recommended IND'] == label]['token_count']
    ax.hist(subset, bins=30, alpha=0.6, label=f'Recommended={label}', color=color)
ax.set_title('Token Count by Recommendation Label')
ax.set_xlabel('Tokens after preprocessing')
ax.legend()
plt.show()

## Hold-out Evaluation

80/20 stratified split. Production model: **CountVectorizer (min_df=2) + Logistic Regression**.

In [ ]:
holdout = evaluate_holdout(df)
print(f"Accuracy:  {holdout['accuracy']:.1%}")
print(f"Precision: {holdout['precision']:.1%}")
print(f"Recall:    {holdout['recall']:.1%}")
print(f"F1:        {holdout['f1']:.1%}")
print('Confusion matrix:', holdout['confusion_matrix'])
print()
print(holdout['classification_report'])

## Model Comparison

In [ ]:
comparison = compare_models(df)
comparison_df = pd.DataFrame(comparison)[
    ['vectorizer', 'classifier', 'accuracy', 'precision', 'recall', 'f1', 'train_seconds']
]
comparison_df

In [ ]:
plot_df = comparison_df.copy()
plot_df['model'] = plot_df['vectorizer'] + ' + ' + plot_df['classifier']

fig, ax = plt.subplots(figsize=(8, 4))
x = range(len(plot_df))
ax.bar(x, plot_df['accuracy'], color='#3b82f6', alpha=0.85, label='Accuracy')
ax.bar([i + 0.35 for i in x], plot_df['f1'], color='#2ecc71', alpha=0.85, label='F1')
ax.set_xticks([i + 0.175 for i in x])
ax.set_xticklabels(plot_df['model'], rotation=15, ha='right')
ax.set_ylim(0.8, 1.0)
ax.set_title('Model Comparison on Held-out Test Set')
ax.legend()
plt.tight_layout()
plt.show()

## Sample Preprocessing

Shared preprocessing is used for both training and inference to avoid train/serve skew.

In [ ]:
sample = df.iloc[0]
raw = f"{sample['Title']} {sample['Review Text']}"
print('Raw:', raw[:120], '...')
print('Processed:', preprocess(raw)[:120], '...')